# Phase 1 데이터 생성 파이프라인

robosuite/LIBERO 씬 상태(팔 관절각, 물체 pose)를 직접 조작해서 positive/hard-negative 이미지 쌍과 물체 GT를 생성한다.

**이 노트북은 라이브 렌더링이 필요하다** (Phase 0처럼 사전 렌더링된 pkl로 우회할 수 없음 — 새로운 팔 자세/물체 배치를 실시간으로 렌더링해야 하므로).

실행 전: 런타임 유형을 GPU로 설정. **L4로 시작** (Phase 0에서 단일 이미지 forward가 L4로 충분했고, 이 파이프라인도 배치 크기를 키우지 않고 반복만 하는 구조라 동일하게 충분할 것으로 예상 — 속도가 문제되면 그때 A100으로).

In [ ]:
!nvidia-smi

## 1. 설치

`research/phase0.ipynb`에서 검증된 순서와 동일하다.

In [ ]:
!pip install -q torch torchvision torchaudio

In [ ]:
!git clone https://github.com/airhood/openvla-ivm.git
%cd openvla-ivm
!pip install -q -e .

`torch==2.2.0`이 NumPy 2.0 이전 ABI라서 numpy를 1.x로 고정한다 (Phase 0와 동일 이유).

In [ ]:
!pip install -q "numpy<2"

In [ ]:
!git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git
!pip install -q -e LIBERO
!pip install -q -r experiments/robot/libero/libero_requirements.txt

robosuite/gym 설치가 numpy를 다시 2.x로 끌어올릴 수 있어 재고정한다.

In [ ]:
!pip install -q "numpy<2"

이 노트북은 **라이브 렌더링이 필요**해서 Xvfb+glfw가 필수다 (Phase 0는 사전 렌더링 pkl로 이 과정을 건너뛸 수 있었지만, 여기선 그럴 수 없음).

In [ ]:
!apt-get update -qq && apt-get install -y -qq xvfb

## 2. Smoke Test (본 파이프라인 작성 전 API 검증)

`scene_utils.py`는 아직 작성 전이다 — robosuite의 물체/관절 이름 규칙, qpos 직접 조작이 렌더링에 반영되는지를 로컬에서 검증할 수 없었기 때문에, 이 스크립트로 먼저 확인한다.

출력에서 확인할 것:
1. `body_names`/`joint_names` 목록 — 물체 이름 규칙 파악
2. `_ref_joint_pos_indexes`가 정상적으로 나오는지 (FAIL이면 robosuite 버전에 따라 속성명이 다른 것 — 출력된 대안 속성 목록 확인)
3. `01_before.png` vs `02_after_arm_change.png` — 팔 자세를 바꾼 게 실제로 렌더링에 반영됐는지 육안 확인
4. `PASS` 출력 및 이미지 평균 차이 값

In [ ]:
!MUJOCO_GL=glfw xvfb-run -a --server-args="-screen 0 1024x768x24" python research/data_generation/smoke_test.py \
  --task_suite_name libero_spatial \
  --task_id 0

In [ ]:
from PIL import Image
display(Image.open("./research/data_generation/smoke_out/01_before.png"))
display(Image.open("./research/data_generation/smoke_out/02_after_arm_change.png"))

## 3. 다음 단계

위 smoke test가 PASS하고 이미지에서 팔이 실제로 움직인 게 육안으로 확인되면, `scene_utils.py`의 나머지 함수(물체 pose get/set, positive/hard-negative 생성)를 이 검증된 API 위에 작성한다.

`body_names` 출력에서 조작 대상 물체 이름을 확인해서 알려주면, 그 이름으로 물체 pose 조작 부분을 마저 작성한다.